# Leakage hunt

One row is one client in the bank's phone campaign, described as of their last call. `y` says whether they subscribed to a term deposit. The contractor's model uses every column and claims 90% accuracy. The first code cell rebuilds that model; your job starts at step 2.

## Step 1 · Run the starter

The cell loads the table, splits it 75 / 25 with `stratify=y`, and defines `fit_and_score(columns)`. That function fits the contractor's pipeline (scale the numbers, one-hot encode the text, logistic regression) on the columns you pass and prints two numbers on the test rows:

- **accuracy**: the share of test clients the model gets right;
- **hit rate**: the model ranks the test clients, the bank calls the top 10%, and the hit rate is the share of those calls that reach a subscriber.

Depending on the download route, `unknown` arrives as text or as a missing value; the starter turns missing values back into the category `"unknown"`.

In [1]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

df = pd.read_csv("data/bank_marketing.csv")
df = df.rename(columns={"deposit": "y", "day": "day_of_week"})    # one set of names for every download route
for col in ["job", "education", "contact", "poutcome"]:
    df[col] = df[col].fillna("unknown")

y = (df["y"] == "yes").astype(int)
X = df.drop(columns="y")
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)
CALLS = len(y_test) // 10                     # the bank calls the top 10% of test clients


def fit_and_score(columns, train=X_train, test=X_test):
    """Fit the contractor's pipeline on these columns; print accuracy and the call-list hit rate."""
    num = [c for c in columns if pd.api.types.is_numeric_dtype(train[c])]
    cat = [c for c in columns if c not in num]
    pre = ColumnTransformer([("num", StandardScaler(), num),
                             ("cat", OneHotEncoder(handle_unknown="ignore"), cat)])
    model = make_pipeline(pre, LogisticRegression(max_iter=2000)).fit(train[columns], y_train)
    proba = model.predict_proba(test[columns])[:, 1]
    called = np.argsort(-proba)[:CALLS]       # the clients the model would call first
    acc = accuracy_score(y_test, model.predict(test[columns]))
    hit = y_test.values[called].mean()
    print(f"accuracy {acc:.3f} | hit rate {hit:.3f} on {CALLS} called clients")
    return model, acc, hit


print(len(X_train), "training rows,", len(X_test), "test rows")
contractor, acc_all, hit_all = fit_and_score(list(X.columns))

33908 training rows, 11303 test rows


accuracy 0.901 | hit rate 0.604 on 1130 called clients


## Step 2 · Before or after the call?

The director asks: "Can we trust that number for next month's calls?" For each column, decide whether its value exists **before** the call you are deciding to make. The README's column table says what each one holds. Write your list in the next cell.

**Sample answer.** Every column describes the client or earlier contact, so it exists before the call, except `duration`: the length of the call is known only after the call ends. The UCI variable notes say the same and ask for `duration` to be discarded for a realistic model. (`campaign`, `day_of_week` and `month` are the bonus question.)

In [2]:
after = ["duration"]
before = [c for c in X.columns if c not in after]
print("after the call:", after)
print("before the call:", before)

after the call: ['duration']
before the call: ['age', 'job', 'marital', 'education', 'default', 'balance', 'housing', 'loan', 'contact', 'day_of_week', 'month', 'campaign', 'pdays', 'previous', 'poutcome']


## Step 3 · Remove the *after* columns and refit

Call `fit_and_score` with the *before* columns only.

In [3]:
honest, acc_before, hit_before = fit_and_score(before)

accuracy 0.895 | hit rate 0.496 on 1130 called clients


## Step 4 · A model that knows nothing

Score `DummyClassifier(strategy="most_frequent")` on the same split and compare its accuracy with step 1. Also print the share of subscribers in the test rows: that is the hit rate of calling clients at random.

In [4]:
dummy = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
dummy_acc = accuracy_score(y_test, dummy.predict(X_test))
print(f"dummy accuracy {dummy_acc:.3f}")
print(f"random calls hit {y_test.mean():.3f}")

dummy accuracy 0.883
random calls hit 0.117


## Step 5 · The `pdays` code

What share of rows has `pdays = -1`? A model reads `-1` as "one day less than 0". Add a column `contacted_before` (`yes` / `no`), set `pdays` to 0 where it was `-1`, and refit without the *after* columns.

In [5]:
never = df["pdays"] == -1
print(f"{never.sum()} rows have pdays = -1 ({never.mean():.3f} of all rows)")


def flag_pdays(d):
    d = d.copy()
    d["contacted_before"] = np.where(d["pdays"] == -1, "no", "yes")
    d["pdays"] = d["pdays"].where(d["pdays"] != -1, 0)
    return d


Xf_train, Xf_test = flag_pdays(X_train), flag_pdays(X_test)
flagged, _, _ = fit_and_score(before + ["contacted_before"], Xf_train, Xf_test)

36954 rows have pdays = -1 (0.817 of all rows)


accuracy 0.895 | hit rate 0.497 on 1130 called clients


## Step 6 · Two sentences for the director

What the contractor's score was, what the honest score is, and why the gap exists. Write them in a markdown cell, or print them.

In [6]:
print(f"With every column, {hit_all:.1%} of the clients on the model's call list subscribe, but the model "
      f"used the call length, which nobody knows before dialing. Without it, {hit_before:.1%} of the call list "
      f"subscribe, still well above the {y_test.mean():.1%} that random calls reach, so that is the number to plan with.")

With every column, 60.4% of the clients on the model's call list subscribe, but the model used the call length, which nobody knows before dialing. Without it, 49.6% of the call list subscribe, still well above the 11.7% that random calls reach, so that is the number to plan with.


## Check yourself

In [7]:
check = {
    "Step 1 · test rows": len(X_test),
    "Step 1 · accuracy, every column": round(acc_all, 3),
    "Step 1 · hit rate, every column": round(hit_all, 3),
    "Step 1 · called clients": CALLS,
    "Step 2 · after columns": ", ".join(after),
    "Step 3 · hit rate without duration": round(hit_before, 3),
    "Step 3 · accuracy without duration": round(acc_before, 3),
    "Step 4 · dummy accuracy": round(dummy_acc, 3),
    "Step 4 · random calls hit": round(y_test.mean(), 3),
    "Step 5 · rows with pdays = -1": int(never.sum()),
    "Step 5 · share of all rows": round(never.mean(), 3),
}
for label, value in check.items():
    print(f"{label:<40} {value}")

Step 1 · test rows                       11303
Step 1 · accuracy, every column          0.901
Step 1 · hit rate, every column          0.604
Step 1 · called clients                  1130
Step 2 · after columns                   duration
Step 3 · hit rate without duration       0.496
Step 3 · accuracy without duration       0.895
Step 4 · dummy accuracy                  0.883
Step 4 · random calls hit                0.117
Step 5 · rows with pdays = -1            36954
Step 5 · share of all rows               0.817
